# Day 28 - Balanced trees in one go: AVL, red-black, B-tree

Code: [day 28 on GitHub](https://github.com/KingRei/100DaysPython/tree/master/day%2028%20-%20balanced%20trees)

A binary search tree is only as good as the order its keys arrive in. Feed it
sorted ids and it turns into a linked list. This notebook builds the three
classic answers - AVL, red-black and B-tree - on top of one shared primitive,
the rotation, and measures what each of them actually costs.

## 1. The problem: a BST is a linked list in disguise

Insertion order decides the shape, and sorted input is the common case: primary
keys, timestamps, a restored database dump. The tree below has no bug in it -
it is just unlucky.

In [1]:
import random


class Node:
    """A plain BST node."""
    __slots__ = ('key', 'left', 'right')

    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None


def bst_insert(root, key):
    """Iterative on purpose: a sorted run would blow the recursion limit."""
    node = Node(key)
    if root is None:
        return node
    cur = root
    while True:
        if key < cur.key:
            if cur.left is None:
                cur.left = node
                return root
            cur = cur.left
        elif key > cur.key:
            if cur.right is None:
                cur.right = node
                return root
            cur = cur.right
        else:
            return root


def height(node):
    """Height in edges, iteratively."""
    best, stack = -1, [(node, 0)]
    while stack:
        n, d = stack.pop()
        if n is None:
            best = max(best, d - 1)
            continue
        stack.append((n.left, d + 1))
        stack.append((n.right, d + 1))
    return best


def mean_cost(root, keys):
    """Average number of comparisons a successful lookup needs."""
    total = 0
    for k in keys:
        cur, steps = root, 0
        while cur is not None:
            steps += 1
            if k == cur.key:
                break
            cur = cur.left if k < cur.key else cur.right
        total += steps
    return total / len(keys)


keys = list(range(1, 2048))
sorted_tree = None
for k in keys:
    sorted_tree = bst_insert(sorted_tree, k)

shuffled = keys[:]
random.Random(28).shuffle(shuffled)
shuffled_tree = None
for k in shuffled:
    shuffled_tree = bst_insert(shuffled_tree, k)

print('sorted   : height %5d   mean lookup %8.2f'
      % (height(sorted_tree), mean_cost(sorted_tree, keys)))
print('shuffled : height %5d   mean lookup %8.2f'
      % (height(shuffled_tree), mean_cost(shuffled_tree, keys)))
print('optimal  : height %5d' % 10)

sorted   : height  2046   mean lookup  1024.00
shuffled : height    22   mean lookup    12.62
optimal  : height    10


## 2. The rotation

Every balanced binary tree in this notebook repairs itself with the same move.
A rotation changes three pointers and nothing else; the in-order sequence
`A x B y C` reads the same before and after, so the search property survives.

In [2]:
class AVLNode:
    __slots__ = ('key', 'left', 'right', 'h')

    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.h = 0


def node_height(n):
    return -1 if n is None else n.h


def update_height(n):
    n.h = 1 + max(node_height(n.left), node_height(n.right))


def balance_factor(n):
    return node_height(n.left) - node_height(n.right)


def rotate_right(y):
    x = y.left
    y.left = x.right
    x.right = y
    update_height(y)          # the lower node first - its children moved
    update_height(x)
    return x


def rotate_left(x):
    y = x.right
    x.right = y.left
    y.left = x
    update_height(x)
    update_height(y)
    return y


def inorder(n):
    out, stack = [], []
    while stack or n is not None:
        while n is not None:
            stack.append(n)
            n = n.left
        n = stack.pop()
        out.append(n.key)
        n = n.right
    return out


y = AVLNode(20)
y.left = AVLNode(10)
y.right = AVLNode(30)
y.left.left = AVLNode(5)
y.left.right = AVLNode(15)
for n in (y.left.left, y.left.right, y.right, y.left, y):
    update_height(n)

before = inorder(y)
x = rotate_right(y)
print('before rotate_right :', before, ' root', y.key)
print('after  rotate_right :', inorder(x), ' root', x.key)
print('same reading order  :', before == inorder(x))

before rotate_right : [5, 10, 15, 20, 30]  root 20
after  rotate_right : [5, 10, 15, 20, 30]  root 10
same reading order  : True


## 3. AVL: keep every subtree within one level of itself

The invariant is `|height(left) - height(right)| <= 1`. After an insert only the
nodes on the path back to the root can be out of balance, and there are exactly
four shapes to repair. Which one you are in is decided by the sign of the
*child's* balance factor: same sign as the parent means a single rotation,
opposite sign means the zig-zag case and two rotations.

In [3]:
def avl_rebalance(n, stats=None):
    update_height(n)
    bf = balance_factor(n)
    if bf > 1:                                # left heavy
        if balance_factor(n.left) < 0:        # LR - zig-zag
            n.left = rotate_left(n.left)
            if stats is not None:
                stats['double'] += 1
        elif stats is not None:
            stats['single'] += 1
        return rotate_right(n)
    if bf < -1:                               # right heavy
        if balance_factor(n.right) > 0:       # RL - zig-zag
            n.right = rotate_right(n.right)
            if stats is not None:
                stats['double'] += 1
        elif stats is not None:
            stats['single'] += 1
        return rotate_left(n)
    return n


def avl_insert(root, key, stats=None):
    """Iterative: walk down, then rebalance back up along the recorded path."""
    if root is None:
        return AVLNode(key)
    path = []
    cur = root
    while True:
        if key < cur.key:
            if cur.left is None:
                cur.left = AVLNode(key)
                break
            path.append((cur, True))
            cur = cur.left
        elif key > cur.key:
            if cur.right is None:
                cur.right = AVLNode(key)
                break
            path.append((cur, False))
            cur = cur.right
        else:
            return root
    node = avl_rebalance(cur, stats)
    while path:
        parent, went_left = path.pop()
        if went_left:
            parent.left = node
        else:
            parent.right = node
        node = avl_rebalance(parent, stats)
    return node


for order, name in [([30, 20, 10], 'LL'), ([10, 20, 30], 'RR'),
                    ([30, 10, 20], 'LR'), ([10, 30, 20], 'RL')]:
    st = {'single': 0, 'double': 0}
    t = None
    for k in order:
        t = avl_insert(t, k, st)
    print('%s  inserted %-12s -> root %d, height %d, %s rotation'
          % (name, str(order), t.key, node_height(t),
             'single' if st['single'] else 'double'))

LL  inserted [30, 20, 10] -> root 20, height 1, single rotation
RR  inserted [10, 20, 30] -> root 20, height 1, single rotation
LR  inserted [30, 10, 20] -> root 20, height 1, double rotation
RL  inserted [10, 30, 20] -> root 20, height 1, double rotation


## 4. What the invariant buys

Same 100000 sorted keys. The AVL tree stays 16 levels deep instead of 99999,
and it pays for that with a rotation on almost every insert - which is exactly
what you want, because a rotation is three pointer writes and a lookup that
misses is a cache miss per level.

In [4]:
n = 100000
big_keys = list(range(n))
probes = random.Random(2828).sample(big_keys, 2000)

stats = {'single': 0, 'double': 0}
avl = None
for k in big_keys:
    avl = avl_insert(avl, k, stats)

print('AVL over %d sorted keys' % n)
print('  height       %d' % node_height(avl))
print('  mean lookup  %.2f comparisons' % mean_cost(avl, probes))
print('  rotations    %d (%d single + %d double)'
      % (stats['single'] + 2 * stats['double'],
         stats['single'], stats['double']))
import math
print('  log2(n)      %.1f  - the floor no BST can beat' % math.log2(n))

AVL over 100000 sorted keys
  height       16
  mean lookup  15.72 comparisons
  rotations    99983 (99983 single + 0 double)
  log2(n)      16.6  - the floor no BST can beat


## 5. Red-black: five properties instead of a height rule

A red-black tree does not measure heights. It colours nodes and keeps five
properties, the load-bearing pair being *a red node has two black children* and
*every root-to-leaf path contains the same number of black nodes*. Together
they force the longest path to be at most twice the shortest, which is a weaker
promise than AVL's - and cheaper to maintain.

In [5]:
RED, BLACK = 'R', 'B'


class RBNode:
    __slots__ = ('key', 'color', 'left', 'right', 'parent')

    def __init__(self, key, color=RED):
        self.key = key
        self.color = color
        self.left = NIL
        self.right = NIL
        self.parent = NIL


NIL = RBNode.__new__(RBNode)       # built by hand: __init__ needs NIL to exist
NIL.key = None
NIL.color = BLACK
NIL.left = NIL.right = NIL.parent = NIL


class RedBlackTree:
    def __init__(self):
        self.root = NIL
        self.stats = {'rotations': 0, 'recolors': 0}

    def _rotate_left(self, x):
        y = x.right
        x.right = y.left
        if y.left is not NIL:
            y.left.parent = x
        y.parent = x.parent
        if x.parent is NIL:
            self.root = y
        elif x is x.parent.left:
            x.parent.left = y
        else:
            x.parent.right = y
        y.left = x
        x.parent = y
        self.stats['rotations'] += 1

    def _rotate_right(self, y):
        x = y.left
        y.left = x.right
        if x.right is not NIL:
            x.right.parent = y
        x.parent = y.parent
        if y.parent is NIL:
            self.root = x
        elif y is y.parent.right:
            y.parent.right = x
        else:
            y.parent.left = x
        x.right = y
        y.parent = x
        self.stats['rotations'] += 1

    def insert(self, key):
        z = RBNode(key)
        parent, cur = NIL, self.root
        while cur is not NIL:
            parent = cur
            if key < cur.key:
                cur = cur.left
            elif key > cur.key:
                cur = cur.right
            else:
                return
        z.parent = parent
        if parent is NIL:
            self.root = z
        elif key < parent.key:
            parent.left = z
        else:
            parent.right = z
        self._insert_fixup(z)

    def _insert_fixup(self, z):
        while z.parent.color == RED:
            gp = z.parent.parent
            if z.parent is gp.left:
                uncle = gp.right
                if uncle.color == RED:            # case 1: recolour only
                    z.parent.color = uncle.color = BLACK
                    gp.color = RED
                    self.stats['recolors'] += 3
                    z = gp                        # violation moves up two
                else:
                    if z is z.parent.right:       # case 2: make it a line
                        z = z.parent
                        self._rotate_left(z)
                    z.parent.color = BLACK        # case 3: rotate and stop
                    gp.color = RED
                    self.stats['recolors'] += 2
                    self._rotate_right(gp)
            else:
                uncle = gp.left
                if uncle.color == RED:
                    z.parent.color = uncle.color = BLACK
                    gp.color = RED
                    self.stats['recolors'] += 3
                    z = gp
                else:
                    if z is z.parent.left:
                        z = z.parent
                        self._rotate_right(z)
                    z.parent.color = BLACK
                    gp.color = RED
                    self.stats['recolors'] += 2
                    self._rotate_left(gp)
        if self.root.color == RED:
            self.root.color = BLACK
            self.stats['recolors'] += 1

    def height(self):
        best, stack = -1, [(self.root, 0)]
        while stack:
            node, d = stack.pop()
            if node is NIL:
                best = max(best, d - 1)
                continue
            stack.append((node.left, d + 1))
            stack.append((node.right, d + 1))
        return best

    def search_cost(self, key):
        steps, cur = 0, self.root
        while cur is not NIL:
            steps += 1
            if key == cur.key:
                return steps
            cur = cur.left if key < cur.key else cur.right
        return steps


rb = RedBlackTree()
for k in big_keys:
    rb.insert(k)

print('red-black over %d sorted keys' % n)
print('  height        %d' % rb.height())
print('  mean lookup   %.2f comparisons'
      % (sum(rb.search_cost(k) for k in probes) / len(probes)))
print('  rotations     %d' % rb.stats['rotations'])
print('  recolourings  %d' % rb.stats['recolors'])

red-black over 100000 sorted keys
  height        30
  mean lookup   16.09 comparisons
  rotations     99969
  recolourings  499846


## 6. The comparison that actually matters

On the averages the two look interchangeable. The difference shows up in the
*worst single operation*: an AVL insert stops after one rebalance, but an AVL
delete makes a subtree shorter, so the repair can cascade all the way to the
root. Red-black caps both. That bound is why the Linux kernel, `std::map` and
Java's `TreeMap` all pick red-black - a scheduler cannot afford a long tail.

The delete paths are long enough that the notebook imports them from
`balanced_trees.py` rather than repeating them here.

In [6]:
import balanced_trees as B

rng = random.Random(4095)
wkeys = rng.sample(range(1000000), 4095)
victims = rng.sample(wkeys, 2000)

avl2, a_ins, a_del = None, 0, 0
for k in wkeys:
    st = {'single': 0, 'double': 0}
    avl2 = B.avl_insert(avl2, k, st)
    a_ins = max(a_ins, st['single'] + 2 * st['double'])
for k in victims:
    st = {'single': 0, 'double': 0}
    avl2 = B.avl_delete(avl2, k, st)
    a_del = max(a_del, st['single'] + 2 * st['double'])

rb2, r_ins, r_del = B.RedBlackTree(), 0, 0
for k in wkeys:
    before = rb2.stats['rotations']
    rb2.insert(k)
    r_ins = max(r_ins, rb2.stats['rotations'] - before)
for k in victims:
    before = rb2.stats['rotations']
    rb2.delete(k)
    r_del = max(r_del, rb2.stats['rotations'] - before)

print('worst rotations in one operation, over 4095 inserts + 2000 deletes')
print('  AVL        insert %d   delete %d' % (a_ins, a_del))
print('  red-black  insert %d   delete %d' % (r_ins, r_del))
print()
print('both trees still hold the same keys:',
      B.inorder(avl2) == rb2.inorder())

worst rotations in one operation, over 4095 inserts + 2000 deletes
  AVL        insert 2   delete 5
  red-black  insert 2   delete 3

both trees still hold the same keys: True


## 7. B-tree: change the cost model, not the balance rule

AVL and red-black both count comparisons. A database counts *page reads*, and a
page read is a hundred thousand times more expensive than a comparison. So the
B-tree stops storing one key per node and stores as many as fit in a disk page.
With 4 KB pages and 16-byte entries that is 255 keys per node, which drops the
height of a million-key index to 2.

The tree also grows differently: a full child is split *on the way down*, the
median key moves up into the parent, and the only node that ever gains a level
is the root - which is why all the leaves stay at exactly the same depth.

In [7]:
class BTreeNode:
    __slots__ = ('keys', 'children', 'leaf')

    def __init__(self, leaf=True):
        self.keys = []
        self.children = []
        self.leaf = leaf


class BTree:
    """Minimum degree t: every node except the root holds t-1 .. 2t-1 keys."""

    def __init__(self, t=3):
        self.t = t
        self.root = BTreeNode(leaf=True)
        self.splits = 0

    def search(self, key):
        node, visits = self.root, 0
        while node is not None:
            visits += 1
            i = 0
            while i < len(node.keys) and key > node.keys[i]:
                i += 1
            if i < len(node.keys) and key == node.keys[i]:
                return True, visits
            node = None if node.leaf else node.children[i]
        return False, visits

    def insert(self, key):
        root = self.root
        if len(root.keys) == 2 * self.t - 1:      # split the root first
            new = BTreeNode(leaf=False)
            new.children.append(root)
            self.root = new
            self._split_child(new, 0)
            root = new
        self._insert_nonfull(root, key)

    def _split_child(self, parent, i):
        t = self.t
        full = parent.children[i]
        right = BTreeNode(leaf=full.leaf)
        median = full.keys[t - 1]
        right.keys = full.keys[t:]
        full.keys = full.keys[:t - 1]
        if not full.leaf:
            right.children = full.children[t:]
            full.children = full.children[:t]
        parent.keys.insert(i, median)             # the median moves up
        parent.children.insert(i + 1, right)
        self.splits += 1

    def _insert_nonfull(self, node, key):
        while True:
            i = len(node.keys) - 1
            if node.leaf:
                while i >= 0 and key < node.keys[i]:
                    i -= 1
                if i >= 0 and node.keys[i] == key:
                    return
                node.keys.insert(i + 1, key)
                return
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            if len(node.children[i].keys) == 2 * self.t - 1:
                self._split_child(node, i)
                if key > node.keys[i]:
                    i += 1
                elif key == node.keys[i]:
                    return
            node = node.children[i]

    def height(self):
        h, node = 0, self.root
        while not node.leaf:
            node = node.children[0]
            h += 1
        return h

    def node_count(self):
        total, stack = 0, [self.root]
        while stack:
            node = stack.pop()
            total += 1
            stack.extend(node.children)
        return total

    def inorder(self):
        out = []

        def walk(node):
            for i, k in enumerate(node.keys):
                if not node.leaf:
                    walk(node.children[i])
                out.append(k)
            if not node.leaf:
                walk(node.children[-1])

        walk(self.root)
        return out


small = BTree(t=2)
for k in range(1, 11):
    small.insert(k)
print('t=2, keys 1..10 -> root %s, height %d, %d nodes, %d splits'
      % (small.root.keys, small.height(), small.node_count(), small.splits))
print('in order:', small.inorder())

t=2, keys 1..10 -> root [4], height 2, 8 nodes, 5 splits
in order: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [8]:
def degree_for_page(page_bytes=4096, entry_bytes=16):
    """How many keys fit in one page, expressed as a minimum degree."""
    return max(2, page_bytes // entry_bytes // 2)


t = degree_for_page(4096, 16)
big = BTree(t=t)
for k in range(1000000):
    big.insert(k)

found, visits = big.search(999999)
print('t = %d  (up to %d keys per node)' % (t, 2 * t - 1))
print('1,000,000 keys -> height %d, %d nodes' % (big.height(),
                                                 big.node_count()))
print('a lookup reads %d pages; a balanced binary tree would read about 20'
      % visits)

t = 128  (up to 255 keys per node)
1,000,000 keys -> height 2, 7874 nodes
a lookup reads 3 pages; a balanced binary tree would read about 20


## 8. LeetCode 110, 108 and 1382

`110 Balanced Binary Tree` is the AVL invariant with the tree handed to you.
The naive solution calls `height` inside `height` and costs O(n^2); returning a
sentinel from one post-order pass makes it O(n) - that rewrite is the whole
interview.

`108` builds the shortest possible tree from sorted input, and `1382` is the
repair job: read the tree in order, rebuild it from the middle out. The
in-place alternative is the Day-Stout-Warren algorithm, which flattens the tree
into a right-leaning vine and rotates it back down in O(1) extra space.

In [9]:
def is_balanced(root):
    """LC 110 - one post-order pass, -2 means 'already unbalanced'."""

    def check(n):
        if n is None:
            return -1
        lh = check(n.left)
        if lh == -2:
            return -2
        rh = check(n.right)
        if rh == -2 or abs(lh - rh) > 1:
            return -2
        return 1 + max(lh, rh)

    return check(root) != -2


def sorted_array_to_bst(nums):
    """LC 108 - the middle element is the only choice that stays balanced."""

    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        node = Node(nums[mid])
        node.left = build(lo, mid - 1)
        node.right = build(mid + 1, hi)
        return node

    return build(0, len(nums) - 1)


def balance_bst(root):
    """LC 1382 - read in order, rebuild from the middle."""
    return sorted_array_to_bst(inorder(root))


chain = Node(1)
cur = chain
for k in range(2, 64):
    cur.right = Node(k)
    cur = cur.right

print('chain of 63     : height %d, balanced? %s'
      % (height(chain), is_balanced(chain)))
fixed = balance_bst(chain)
print('after rebuilding: height %d, balanced? %s'
      % (height(fixed), is_balanced(fixed)))
print('keys untouched  :', inorder(fixed) == list(range(1, 64)))

chain of 63     : height 62, balanced? False
after rebuilding: height 5, balanced? True
keys untouched  : True


## 9. Tests

If any of these fail, something above is wrong.

In [10]:
# rotations preserve the reading order
t1 = AVLNode(20)
t1.left, t1.right = AVLNode(10), AVLNode(30)
for x in (t1.left, t1.right, t1):
    update_height(x)
assert inorder(t1) == inorder(rotate_right(t1))

# AVL keeps its invariant and its keys
sample = random.Random(7).sample(range(10000), 2000)
tree = None
for k in sample:
    tree = avl_insert(tree, k)
assert inorder(tree) == sorted(sample)


def avl_ok(n):
    if n is None:
        return -1
    lh, rh = avl_ok(n.left), avl_ok(n.right)
    assert abs(lh - rh) <= 1, 'AVL invariant broken at %s' % n.key
    assert n.h == 1 + max(lh, rh)
    return n.h


avl_ok(tree)

# red-black: root black, no red-red, equal black height, height <= 2 log2(n+1)
rbt = RedBlackTree()
for k in sample:
    rbt.insert(k)
assert rbt.root.color == BLACK


def rb_black_height(n):
    if n is NIL:
        return 0
    if n.color == RED:
        assert n.left.color == BLACK and n.right.color == BLACK
    lb, rbh = rb_black_height(n.left), rb_black_height(n.right)
    assert lb == rbh, 'black heights differ'
    return lb + (1 if n.color == BLACK else 0)


rb_black_height(rbt.root)
assert rbt.height() <= 2 * math.log2(len(sample) + 1)

# the worst-case bounds the whole section 6 rests on
assert a_ins <= 2 and r_ins <= 2
assert r_del <= 3

# B-tree: sorted, all leaves at one depth, arity within bounds
bt = BTree(t=3)
for k in random.Random(9).sample(range(5000), 3000):
    bt.insert(k)
assert bt.inorder() == sorted(set(bt.inorder()))
depths = []
stack = [(bt.root, 0)]
while stack:
    node, d = stack.pop()
    if node.leaf:
        depths.append(d)
    else:
        assert len(node.children) == len(node.keys) + 1
        stack.extend((c, d + 1) for c in node.children)
assert len(set(depths)) == 1, 'leaves are not at the same depth'

# LeetCode
assert is_balanced(sorted_array_to_bst(list(range(15)))) is True
assert is_balanced(chain) is False
assert degree_for_page(4096, 16) == 128

print('all assertions passed')

all assertions passed
